In [0]:
%sql
CREATE OR REPLACE TABLE 03_gold_catalog.analytical_tables.dim_customers AS
SELECT
    customer_id,
    customer_name,
    customer_type,
    country,
    signup_date
FROM 02_silver_catalog.transformed_tables.customers;

CREATE OR REPLACE TABLE 03_gold_catalog.analytical_tables.dim_products AS
SELECT
    product_id,
    product_name,
    category,
    cost_price
FROM 02_silver_catalog.transformed_tables.products;

CREATE OR REPLACE TABLE 03_gold_catalog.analytical_tables.dim_regions AS
SELECT
    region_code,
    region_name,
    country
FROM 02_silver_catalog.transformed_tables.regions;

In [0]:
%sql
CREATE OR REPLACE TABLE 03_gold_catalog.analytical_tables.fact_sales AS
SELECT
    ili.invoice_line_id,
    ili.invoice_id,
    ili.product_id,
    i.customer_id,
    i.invoice_date,
    i.region,
    i.currency,
    i.invoice_status,
    ili.quantity,
    ili.unit_price,
    ili.discount,
    ROUND(ili.quantity * ili.unit_price, 2) AS gross_amount,
    ROUND((ili.quantity * ili.unit_price) * ili.discount, 2) AS discount_amount,
    ROUND((ili.quantity * ili.unit_price) * (1 - ili.discount), 2) AS net_amount_local,
    er.rate_to_usd,
    ROUND((ili.quantity * ili.unit_price) * (1 - ili.discount) * er.rate_to_usd, 2) AS revenue_usd,
    p.cost_price,
    ROUND(ili.quantity * p.cost_price, 2) AS cost_total,
    ROUND(((ili.quantity * ili.unit_price) * (1 - ili.discount)) - (ili.quantity * p.cost_price), 2) AS profit_local,
    ROUND((((ili.quantity * ili.unit_price) * (1 - ili.discount)) - (ili.quantity * p.cost_price)) * er.rate_to_usd, 2) AS profit_usd
FROM 02_silver_catalog.transformed_tables.invoice_line_items ili
JOIN 02_silver_catalog.transformed_tables.invoices i
    ON ili.invoice_id = i.invoice_id
JOIN 02_silver_catalog.transformed_tables.products p
    ON ili.product_id = p.product_id
LEFT JOIN 02_silver_catalog.transformed_tables.exchange_rates er
    ON i.currency = er.currency
    AND i.invoice_date = er.date;

In [0]:
%sql
CREATE OR REPLACE TABLE 03_gold_catalog.analytical_tables.fact_payments AS
SELECT
    p.payment_id,
    p.invoice_id,
    i.customer_id,
    p.payment_date,
    p.payment_amount,
    p.payment_method,
    i.currency,
    i.region,
    er.rate_to_usd,
    ROUND(p.payment_amount * er.rate_to_usd, 2) AS payment_amount_usd
FROM 02_silver_catalog.transformed_tables.payments p
JOIN 02_silver_catalog.transformed_tables.invoices i
    ON p.invoice_id = i.invoice_id
LEFT JOIN 02_silver_catalog.transformed_tables.exchange_rates er
    ON i.currency = er.currency
    AND p.payment_date = er.date;